### Cell 0: Clean Working Directory

**Why this cell:** Every "Run All" regenerates the dataset, training, and inference outputs 
from scratch. Without clearing old output first, new files pile up alongside or overwrite 
files from the previous run, making the output panel messy.

**Key Concept:** Only wipes `/kaggle/working` (our own scratch space) — `/kaggle/input` 
(the uploaded dataset) is untouched, since it's read-only and lives elsewhere.

**Hyperparameters:** None — cleanup only.

In [1]:
import shutil, os

for f in os.listdir("/kaggle/working"):
    shutil.rmtree(f"/kaggle/working/{f}", ignore_errors=True) if os.path.isdir(f"/kaggle/working/{f}") else os.remove(f"/kaggle/working/{f}")

### Cell 1: Environment Setup

**Why this cell:** This notebook doesn't train anything — it loads your already-trained model 
and runs calibration, tracking, and speed calculation on top of it. Still needs Ultralytics (to 
load and run the YOLO model) and `filterpy` (for the Kalman filter that smooths the ball's 
trajectory across frames).

**Key Concept:** Same two dependencies as notebook 1, since inference and tracking use the same 
core libraries training did — just no training-specific setup needed here.

**Hyperparameters:** None — dependency installation only.

In [2]:
!pip install -q ultralytics opencv-python-headless filterpy

import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

import filterpy
print(f"filterpy installed OK")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 12.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 7.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics version: 8.4.158
filterpy installed OK


### Cell 2: Imports & Configuration

**Why this cell:** Centralizes imports and project-wide constants for this notebook — this 
time pointed at loading a pre-trained model rather than training one. `MODEL_PATH` and 
`VIDEO_PATH` are left unset here and filled in Cell 3 once we confirm exactly where Kaggle 
mounted the uploaded model and video datasets.

**Key Concept:** Same real-world cricket geometry constants as before — 20.12 m pitch length, 
1.22 m popping crease — since the speed math depends on them regardless of which notebook is 
running it.

**Hyperparameters:** `PITCH_LENGTH_METERS = 20.12`, `POPPING_CREASE_DISTANCE_M = 1.22` (fixed, 
not tunable). `MAX_BALL_SPEED_KMH = 160` — sanity ceiling above the fastest deliveries ever 
recorded, so genuinely fast balls are never rejected.

In [3]:
import os
import re
import cv2
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from filterpy.kalman import KalmanFilter

SEED = 42
np.random.seed(SEED)

WORKING_DIR = "/kaggle/working"
MODEL_PATH = None   # set in Cell 3
VIDEO_PATH = None   # set in Cell 3

PITCH_LENGTH_METERS = 20.12
POPPING_CREASE_DISTANCE_M = 1.22
MAX_BALL_SPEED_KMH = 160

os.makedirs(WORKING_DIR, exist_ok=True)
print("Config loaded.")

Config loaded.


### Cell 3: Locate Trained Model & C6806 Video

**Why this cell:** Sets both paths this notebook depends on — the trained detector (uploaded 
as a Kaggle Model, so it mounts at a different path pattern than a Dataset) and the C6806 video 
(from the same dataset used in notebook 1). Verifying both exist here catches a path typo 
immediately, rather than failing deep inside a later cell.

**Key Concept:** Kaggle Models and Kaggle Datasets mount under different path structures — 
`/kaggle/input/<model-slug>/...` for models vs `/kaggle/input/datasets/<user>/<dataset-slug>/...` 
for datasets. Both are read-only inputs either way; only the path shape differs.

**Hyperparameters:** None — path configuration only.

In [4]:
MODEL_PATH = "/kaggle/input/datasets/muhammadhaseebnew/cricket-ball-detector-c6806/ball_detector_c6806_best.pt"
VIDEO_PATH = "/kaggle/input/datasets/muhammadhaseebnew/cricket-data-train/cricket_data/C6806.mp4"

assert os.path.exists(MODEL_PATH), f"Model not found at {MODEL_PATH}"
assert os.path.exists(VIDEO_PATH), f"Video not found at {VIDEO_PATH}"

print(f"Model: {MODEL_PATH}")
print(f"Video: {VIDEO_PATH}")

cap = cv2.VideoCapture(VIDEO_PATH)
VIDEO_FPS = cap.get(cv2.CAP_PROP_FPS)
VIDEO_W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
VIDEO_H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
VIDEO_TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(f"\nResolution: {VIDEO_W}x{VIDEO_H}, FPS: {VIDEO_FPS:.2f}, Total frames: {VIDEO_TOTAL_FRAMES}")

Model: /kaggle/input/datasets/muhammadhaseebnew/cricket-ball-detector-c6806/ball_detector_c6806_best.pt
Video: /kaggle/input/datasets/muhammadhaseebnew/cricket-data-train/cricket_data/C6806.mp4

Resolution: 1920x1080, FPS: 30.00, Total frames: 3804


### Cell 4: Load Model & Verify

**Why this cell:** Confirms the saved checkpoint actually loads correctly and can run inference 
before building anything on top of it — calibration and tracking in later cells both depend on 
this working. A quick check on a few frames catches a corrupted/incompatible checkpoint now, 
not buried inside a longer pipeline later.

**Key Concept:** `YOLO(MODEL_PATH)` loads the trained weights directly — no retraining, no 
reference to the training session, just the saved `.pt` file standing on its own.

**Hyperparameters:** `N_CHECK_FRAMES = 5`, `conf=0.25`, `imgsz=1280` (matches training 
resolution) — same quick sanity check pattern used in notebook 1.

In [5]:
detector = YOLO(MODEL_PATH)
print(f"Model loaded: {MODEL_PATH}")
print(f"Classes: {detector.names}")

N_CHECK_FRAMES = 5
cap = cv2.VideoCapture(VIDEO_PATH)
checked = 0
while checked < N_CHECK_FRAMES:
    ret, frame = cap.read()
    if not ret:
        break
    r = detector.predict(frame, imgsz=1280, conf=0.25, verbose=False)[0]
    classes_found = [int(b.cls.item()) for b in r.boxes]
    print(f"Frame {checked}: detected classes {classes_found}")
    checked += 1
cap.release()

print("\nModel loads and detects correctly.")

Model loaded: /kaggle/input/datasets/muhammadhaseebnew/cricket-ball-detector-c6806/ball_detector_c6806_best.pt
Classes: {0: 'WICKET1', 1: 'WICKET2', 2: 'BALL'}
Frame 0: detected classes [0, 1]
Frame 1: detected classes [0, 1]
Frame 2: detected classes [0, 1]
Frame 3: detected classes [0, 1]
Frame 4: detected classes [0, 1]

Model loads and detects correctly.


### Cell 5: Test Detection on Full C6806 Video

**Why this cell:** Runs the model on the complete video to visually confirm detection quality 
before calibration/tracking — moved here from notebook 1 since it's slow (full 3,804-frame 
pass) and belongs with the analysis work, where you can re-run it anytime against the saved 
checkpoint without retraining.

**Key Concept:** `model.predict()` on a video source processes it frame-by-frame and can save 
an annotated output automatically, alongside a results object we can summarize.

**Hyperparameters:** `conf=0.25`, `imgsz=1280` (matches training resolution). `verbose=False` 
to avoid flooding output with one line per frame.

In [6]:
detect_results = detector.predict(
    source=VIDEO_PATH,
    imgsz=1280,
    conf=0.25,
    save=True,
    project=os.path.join(WORKING_DIR, "inference"),
    name="test_c6806",
    exist_ok=True,
    verbose=False,
    stream=True,   # avoids accumulating all results in RAM at once
)

counts = {0: 0, 1: 0, 2: 0}
frames_with_ball = 0
total = 0

for r in detect_results:
    total += 1
    if any(int(b.cls.item()) == 2 for b in r.boxes):
        frames_with_ball += 1
    for b in r.boxes:
        counts[int(b.cls.item())] += 1

print(f"WICKET1: {counts[0]}, WICKET2: {counts[1]}, BALL: {counts[2]}")
print(f"Frames with BALL detected: {frames_with_ball}/{total} ({frames_with_ball/total*100:.1f}%)")

Results saved to /kaggle/working/inference/test_c6806
WICKET1: 3804, WICKET2: 3806, BALL: 2113
Frames with BALL detected: 2088/3804 (54.9%)


### Cell 6: Calibrate Pixel-to-Meter Scale from Wicket Positions

**Why this cell:** Speed calculation needs to convert the ball's movement in *pixels* into 
*meters*. The wickets are static and sit at the two ends of the pitch, a known real-world 
distance apart (`PITCH_LENGTH_METERS`), so averaging their detected positions across several 
frames gives a stable pixel-to-meter scale factor without any manual measurement.

**Key Concept:** Sample a spread of frames (not just the first few, in case of slight camera 
shake or partial occlusion early on), take the mean center of each wicket class's bounding box 
per frame, then average those centers across frames for a robust pixel distance between the 
two wickets.

**Hyperparameters:** `N_CALIB_FRAMES = 30`, sampled evenly across the video (not just the 
start) for robustness against noise in any single frame.

In [7]:
N_CALIB_FRAMES = 30
sample_indices = set(np.linspace(0, VIDEO_TOTAL_FRAMES - 1, N_CALIB_FRAMES, dtype=int).tolist())

cap = cv2.VideoCapture(VIDEO_PATH)
wicket1_centers, wicket2_centers = [], []

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx in sample_indices:
        r = detector.predict(frame, imgsz=1280, conf=0.25, verbose=False)[0]
        for b in r.boxes:
            cls_id = int(b.cls.item())
            x1, y1, x2, y2 = b.xyxy[0].tolist()
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
            if cls_id == 0:
                wicket1_centers.append((cx, cy))
            elif cls_id == 1:
                wicket2_centers.append((cx, cy))
    frame_idx += 1
cap.release()

assert wicket1_centers and wicket2_centers, "Wickets not detected in any sampled frame"

w1_mean = np.mean(wicket1_centers, axis=0)
w2_mean = np.mean(wicket2_centers, axis=0)
pixel_distance = np.linalg.norm(w1_mean - w2_mean)

METERS_PER_PIXEL = PITCH_LENGTH_METERS / pixel_distance

print(f"WICKET1 mean center: {w1_mean}")
print(f"WICKET2 mean center: {w2_mean}")
print(f"Pixel distance between wickets: {pixel_distance:.2f}px")
print(f"Calibration: {METERS_PER_PIXEL:.6f} meters/pixel")

WICKET1 mean center: [     1818.1      716.83]
WICKET2 mean center: [     90.463      742.81]
Pixel distance between wickets: 1727.87px
Calibration: 0.011644 meters/pixel


### Cell 7: Track Ball Position Through a Single Delivery with Kalman Filter

**Why this cell:** Raw per-frame detections are noisy — the ball can be missed for a frame or 
two, or its bounding box center can jitter slightly. A Kalman filter predicts the ball's next 
position based on a constant-velocity motion model and corrects that prediction with each new 
detection, giving a smooth trajectory even through brief gaps in detection.

**Key Concept:** State vector `[x, y, vx, vy]` — position and velocity in both axes. On each 
frame: predict the next state, then if a BALL detection exists, update the filter with the 
measured center; if not, keep the predicted position (coasting through the gap).

**Hyperparameters:** `DELIVERY_START_FRAME = 40` (matches the "~frame 40+" pattern you noted 
in Cell 4), `DELIVERY_END_FRAME = 120` (80-frame window — generous for one delivery at 30fps, 
covering release through impact/pass), initial state uncertainty and process noise set to 
loose defaults since we'll tune them only if the trajectory looks jittery.

In [8]:
DELIVERY_START_FRAME = 40
DELIVERY_END_FRAME = 120

kf = KalmanFilter(dim_x=4, dim_z=2)
kf.F = np.array([[1, 0, 1, 0],
                  [0, 1, 0, 1],
                  [0, 0, 1, 0],
                  [0, 0, 0, 1]])
kf.H = np.array([[1, 0, 0, 0],
                  [0, 1, 0, 0]])
kf.P *= 500.
kf.R *= 5.
kf.Q *= 0.01

cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, DELIVERY_START_FRAME)

trajectory = []
initialized = False

for frame_idx in range(DELIVERY_START_FRAME, DELIVERY_END_FRAME):
    ret, frame = cap.read()
    if not ret:
        break

    r = detector.predict(frame, imgsz=1280, conf=0.25, verbose=False)[0]
    ball_center = None
    for b in r.boxes:
        if int(b.cls.item()) == 2:
            x1, y1, x2, y2 = b.xyxy[0].tolist()
            ball_center = ((x1 + x2) / 2, (y1 + y2) / 2)
            break

    if not initialized:
        if ball_center is None:
            continue
        kf.x = np.array([ball_center[0], ball_center[1], 0, 0])
        initialized = True
    else:
        kf.predict()
        if ball_center is not None:
            kf.update(np.array(ball_center))

    trajectory.append({
        "frame": frame_idx,
        "raw_x": ball_center[0] if ball_center else None,
        "raw_y": ball_center[1] if ball_center else None,
        "kf_x": kf.x[0],
        "kf_y": kf.x[1],
    })

cap.release()

traj_df = pd.DataFrame(trajectory)
print(f"Tracked {len(traj_df)} frames ({DELIVERY_START_FRAME}-{DELIVERY_END_FRAME})")
print(f"Frames with raw detection: {traj_df['raw_x'].notna().sum()}")
traj_df.head(10)

Tracked 80 frames (40-120)
Frames with raw detection: 73


,frame,raw_x,raw_y,kf_x,kf_y
0,40,151.264984,710.603058,151.264984,710.603058
1,41,NaN,NaN,151.264984,710.603058
2,42,NaN,NaN,151.264984,710.603058
3,43,NaN,NaN,151.264984,710.603058
4,44,NaN,NaN,151.264984,710.603058
5,45,NaN,NaN,151.264984,710.603058
6,46,NaN,NaN,151.264984,710.603058
7,47,271.493896,607.983032,271.469856,608.003551
8,48,316.619583,605.448486,310.044733,602.706905
9,49,360.118790,602.655212,355.001672,600.400314


### Cell 8: Calculate Ball Speed from Kalman-Smoothed Trajectory

**Why this cell:** Converts the smoothed pixel trajectory from Cell 7 into a real-world speed 
in km/h, using the `METERS_PER_PIXEL` calibration from Cell 6 and the video's known `VIDEO_FPS`. 
Speed is computed per consecutive frame pair (instantaneous), then also averaged/peaked across 
the delivery for a single headline number.

**Key Concept:** Distance between two consecutive KF positions (in pixels) × `METERS_PER_PIXEL` 
gives meters moved in `1/VIDEO_FPS` seconds. Dividing by that time delta gives m/s, then 
converting to km/h (`× 3.6`). Frame-to-frame speed is noisy even after smoothing, so we care 
most about the **peak speed over a short rolling window** (release speed is what's usually 
reported) rather than any single frame-pair value.

**Hyperparameters:** `ROLLING_WINDOW = 5` frames for smoothing instantaneous speed spikes; 
`MAX_BALL_SPEED_KMH` (from Cell 2) used to flag/reject physically implausible spikes (e.g. from 
a bad detection jump) rather than silently reporting them as the peak.

In [9]:
ROLLING_WINDOW = 5

frame_dt = 1.0 / VIDEO_FPS  # seconds between frames

speeds_kmh = [np.nan]  # no speed for the first frame (no prior point)

for i in range(1, len(traj_df)):
    dx = traj_df.loc[i, "kf_x"] - traj_df.loc[i - 1, "kf_x"]
    dy = traj_df.loc[i, "kf_y"] - traj_df.loc[i - 1, "kf_y"]
    pixel_dist = np.sqrt(dx**2 + dy**2)

    meters = pixel_dist * METERS_PER_PIXEL
    mps = meters / frame_dt
    kmh = mps * 3.6

    speeds_kmh.append(kmh)

traj_df["speed_kmh"] = speeds_kmh

# Flag physically implausible spikes rather than silently trusting them
traj_df["speed_valid"] = traj_df["speed_kmh"] <= MAX_BALL_SPEED_KMH

# Smooth with rolling window to reduce frame-to-frame noise
traj_df["speed_kmh_smooth"] = traj_df["speed_kmh"].rolling(
    window=ROLLING_WINDOW, min_periods=1, center=True
).mean()

valid_speeds = traj_df.loc[traj_df["speed_valid"], "speed_kmh"]

print(f"Frames with implausible speed (> {MAX_BALL_SPEED_KMH} km/h): "
      f"{(~traj_df['speed_valid']).sum()}")
print(f"Peak speed (raw, valid only): {valid_speeds.max():.1f} km/h")
print(f"Peak speed (rolling-smoothed): {traj_df['speed_kmh_smooth'].max():.1f} km/h")
print(f"Mean speed (valid only): {valid_speeds.mean():.1f} km/h")

traj_df[["frame", "raw_x", "raw_y", "kf_x", "kf_y", "speed_kmh", "speed_kmh_smooth"]].head(15)

Frames with implausible speed (> 160 km/h): 2
Peak speed (raw, valid only): 73.4 km/h
Peak speed (rolling-smoothed): 83.9 km/h
Mean speed (valid only): 46.1 km/h


,frame,raw_x,raw_y,kf_x,kf_y,speed_kmh,speed_kmh_smooth
0,40,151.264984,710.603058,151.264984,710.603058,NaN,0.000000
1,41,NaN,NaN,151.264984,710.603058,0.000000,0.000000
2,42,NaN,NaN,151.264984,710.603058,0.000000,0.000000
3,43,NaN,NaN,151.264984,710.603058,0.000000,0.000000
4,44,NaN,NaN,151.264984,710.603058,0.000000,0.000000
5,45,NaN,NaN,151.264984,710.603058,0.000000,39.749444
6,46,NaN,NaN,151.264984,710.603058,0.000000,49.542791
7,47,271.493896,607.983032,271.469856,608.003551,198.747219,60.865187
8,48,316.619583,605.448486,310.044733,602.706905,48.966738,72.391675
9,49,360.118790,602.655212,355.001672,600.400314,56.611980,83.916972


### Cell 9: Generalize Tracking + Speed Calculation into a Reusable Function

**Why this cell:** Cells 7 and 8 hardcoded `DELIVERY_START_FRAME`/`DELIVERY_END_FRAME` and ran 
once. Once automatic delivery segmentation (next cell) detects multiple delivery windows across 
the full video, we need to run the exact same tracking + speed logic per window without 
copy-pasting it. Wrapping it in a function now means Cell 10 just calls this once per detected 
delivery.

**Key Concept:** Same Kalman filter + speed math as before, but parameterized by 
`start_frame`/`end_frame`, returning a clean `DataFrame` (like `traj_df`) plus a small summary 
dict (peak/mean speed) instead of just printing — so results from multiple deliveries can be 
collected and compared later.

**Hyperparameters:** Same defaults as Cells 7-8 (`ROLLING_WINDOW=5`, `conf=0.25`, `imgsz=1280`), 
now as function parameters with those defaults so behavior is unchanged unless overridden.

In [10]:
def track_and_measure_delivery(
    video_path,
    detector,
    start_frame,
    end_frame,
    meters_per_pixel,
    video_fps,
    max_ball_speed_kmh,
    rolling_window=5,
    conf=0.25,
    imgsz=1280,
):
    """
    Tracks the ball across [start_frame, end_frame) using a Kalman filter,
    then computes frame-to-frame speed in km/h.

    Returns:
        traj_df   -- DataFrame with per-frame raw/KF positions and speed
        summary   -- dict with peak/mean speed and detection stats
    """
    kf = KalmanFilter(dim_x=4, dim_z=2)
    kf.F = np.array([[1, 0, 1, 0],
                      [0, 1, 0, 1],
                      [0, 0, 1, 0],
                      [0, 0, 0, 1]])
    kf.H = np.array([[1, 0, 0, 0],
                      [0, 1, 0, 0]])
    kf.P *= 500.
    kf.R *= 5.
    kf.Q *= 0.01

    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    trajectory = []
    initialized = False

    for frame_idx in range(start_frame, end_frame):
        ret, frame = cap.read()
        if not ret:
            break

        r = detector.predict(frame, imgsz=imgsz, conf=conf, verbose=False)[0]
        ball_center = None
        for b in r.boxes:
            if int(b.cls.item()) == 2:
                x1, y1, x2, y2 = b.xyxy[0].tolist()
                ball_center = ((x1 + x2) / 2, (y1 + y2) / 2)
                break

        if not initialized:
            if ball_center is None:
                continue
            kf.x = np.array([ball_center[0], ball_center[1], 0, 0])
            initialized = True
        else:
            kf.predict()
            if ball_center is not None:
                kf.update(np.array(ball_center))

        trajectory.append({
            "frame": frame_idx,
            "raw_x": ball_center[0] if ball_center else None,
            "raw_y": ball_center[1] if ball_center else None,
            "kf_x": kf.x[0],
            "kf_y": kf.x[1],
        })

    cap.release()

    traj_df = pd.DataFrame(trajectory)

    if traj_df.empty:
        return traj_df, {
            "start_frame": start_frame, "end_frame": end_frame,
            "frames_tracked": 0, "frames_detected": 0,
            "peak_speed_kmh": None, "peak_speed_smooth_kmh": None,
            "mean_speed_kmh": None,
        }

    frame_dt = 1.0 / video_fps
    speeds_kmh = [np.nan]
    for i in range(1, len(traj_df)):
        dx = traj_df.loc[i, "kf_x"] - traj_df.loc[i - 1, "kf_x"]
        dy = traj_df.loc[i, "kf_y"] - traj_df.loc[i - 1, "kf_y"]
        pixel_dist = np.sqrt(dx**2 + dy**2)
        mps = (pixel_dist * meters_per_pixel) / frame_dt
        speeds_kmh.append(mps * 3.6)

    traj_df["speed_kmh"] = speeds_kmh
    traj_df["speed_valid"] = traj_df["speed_kmh"] <= max_ball_speed_kmh
    traj_df["speed_kmh_smooth"] = traj_df["speed_kmh"].rolling(
        window=rolling_window, min_periods=1, center=True
    ).mean()

    valid_speeds = traj_df.loc[traj_df["speed_valid"], "speed_kmh"]

    summary = {
        "start_frame": start_frame,
        "end_frame": end_frame,
        "frames_tracked": len(traj_df),
        "frames_detected": int(traj_df["raw_x"].notna().sum()),
        "peak_speed_kmh": float(valid_speeds.max()) if len(valid_speeds) else None,
        "peak_speed_smooth_kmh": float(traj_df["speed_kmh_smooth"].max()) if len(traj_df) else None,
        "mean_speed_kmh": float(valid_speeds.mean()) if len(valid_speeds) else None,
    }

    return traj_df, summary


# Sanity check: re-run on the same window from Cells 7-8, confirm identical results
traj_df, summary = track_and_measure_delivery(
    video_path=VIDEO_PATH,
    detector=detector,
    start_frame=40,
    end_frame=120,
    meters_per_pixel=METERS_PER_PIXEL,
    video_fps=VIDEO_FPS,
    max_ball_speed_kmh=MAX_BALL_SPEED_KMH,
)

print(summary)
traj_df.head(10)

{'start_frame': 40, 'end_frame': 120, 'frames_tracked': 80, 'frames_detected': 73, 'peak_speed_kmh': 73.43563531255107, 'peak_speed_smooth_kmh': 83.91697235200962, 'mean_speed_kmh': 46.09850413637699}


,frame,raw_x,raw_y,kf_x,kf_y,speed_kmh,speed_valid,speed_kmh_smooth
0,40,151.264984,710.603058,151.264984,710.603058,NaN,False,0.000000
1,41,NaN,NaN,151.264984,710.603058,0.000000,True,0.000000
2,42,NaN,NaN,151.264984,710.603058,0.000000,True,0.000000
3,43,NaN,NaN,151.264984,710.603058,0.000000,True,0.000000
4,44,NaN,NaN,151.264984,710.603058,0.000000,True,0.000000
5,45,NaN,NaN,151.264984,710.603058,0.000000,True,39.749444
6,46,NaN,NaN,151.264984,710.603058,0.000000,True,49.542791
7,47,271.493896,607.983032,271.469856,608.003551,198.747219,False,60.865187
8,48,316.619583,605.448486,310.044733,602.706905,48.966738,True,72.391675
9,49,360.118790,602.655212,355.001672,600.400314,56.611980,True,83.916972


### Cell 10a: Automatically Detect Delivery Windows Across Full Video

**Why this cell:** So far every delivery window (40-120) was chosen by hand. A real pipeline 
needs to find delivery boundaries on its own so `track_and_measure_delivery()` can be applied 
to each one automatically, across a video with an unknown number of deliveries.

**Key Concept:** Run detection once across the whole video (frame-by-frame, sequential read — 
no repeated seeking) and record which frames have a BALL detection. Consecutive "ball present" 
frames form a candidate delivery; gaps of `MIN_GAP_FRAMES` or more consecutive "no ball" frames 
mark a boundary between deliveries. Very short blips (a single stray frame with/without 
detection) are smoothed over so noise doesn't fragment a real delivery into several fake ones.

**Hyperparameters:** `MIN_GAP_FRAMES = 15` (0.5s at 30fps) — long enough to not split a delivery 
on a couple of missed detection frames, short enough to actually separate distinct deliveries. 
`MIN_DELIVERY_LENGTH = 20` frames — discards tiny spurious detections (e.g. a single stray false 
positive) that aren't a real delivery.

In [11]:
MIN_GAP_FRAMES = 15
MIN_DELIVERY_LENGTH = 20

cap = cv2.VideoCapture(VIDEO_PATH)
ball_present = []

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    r = detector.predict(frame, imgsz=1280, conf=0.25, verbose=False)[0]
    has_ball = any(int(b.cls.item()) == 2 for b in r.boxes)
    ball_present.append(has_ball)
    frame_idx += 1
cap.release()

ball_present = np.array(ball_present)
print(f"Total frames scanned: {len(ball_present)}")
print(f"Frames with ball: {ball_present.sum()} ({ball_present.mean()*100:.1f}%)")

# Find runs of consecutive "ball present" frames, merging gaps shorter than MIN_GAP_FRAMES
deliveries = []
in_delivery = False
current_start = None
gap_count = 0

for i, present in enumerate(ball_present):
    if present:
        if not in_delivery:
            current_start = i
            in_delivery = True
        gap_count = 0
    else:
        if in_delivery:
            gap_count += 1
            if gap_count >= MIN_GAP_FRAMES:
                delivery_end = i - gap_count + 1
                if delivery_end - current_start >= MIN_DELIVERY_LENGTH:
                    deliveries.append((current_start, delivery_end))
                in_delivery = False
                gap_count = 0

if in_delivery:
    delivery_end = len(ball_present) - gap_count
    if delivery_end - current_start >= MIN_DELIVERY_LENGTH:
        deliveries.append((current_start, delivery_end))

print(f"\nDetected {len(deliveries)} delivery window(s):")
for i, (s, e) in enumerate(deliveries):
    print(f"  Delivery {i+1}: frames {s}-{e} ({e-s} frames, {(e-s)/VIDEO_FPS:.2f}s)")

Total frames scanned: 3804
Frames with ball: 2088 (54.9%)

Detected 19 delivery window(s):
  Delivery 1: frames 40-144 (104 frames, 3.47s)
  Delivery 2: frames 169-263 (94 frames, 3.13s)
  Delivery 3: frames 320-509 (189 frames, 6.30s)
  Delivery 4: frames 654-836 (182 frames, 6.07s)
  Delivery 5: frames 940-1026 (86 frames, 2.87s)
  Delivery 6: frames 1137-1189 (52 frames, 1.73s)
  Delivery 7: frames 1255-1458 (203 frames, 6.77s)
  Delivery 8: frames 1479-1558 (79 frames, 2.63s)
  Delivery 9: frames 1581-1659 (78 frames, 2.60s)
  Delivery 10: frames 1682-1767 (85 frames, 2.83s)
  Delivery 11: frames 1788-1923 (135 frames, 4.50s)
  Delivery 12: frames 2040-2132 (92 frames, 3.07s)
  Delivery 13: frames 2181-2277 (96 frames, 3.20s)
  Delivery 14: frames 2483-2585 (102 frames, 3.40s)
  Delivery 15: frames 2686-2810 (124 frames, 4.13s)
  Delivery 16: frames 2971-3080 (109 frames, 3.63s)
  Delivery 17: frames 3213-3338 (125 frames, 4.17s)
  Delivery 18: frames 3444-3606 (162 frames, 5.40s)


### Cell 10b: Split Anomalously Long Delivery Windows at Their Largest Internal Gap

**Why this cell:** Segmentation in Cell 10 sometimes merges two deliveries into one window 
when CapCut editing compressed the dead time between them below `MIN_GAP_FRAMES`. Rather than 
lowering the global gap threshold (which risks fragmenting genuine single deliveries with a 
brief mid-flight occlusion), this flags only windows that are anomalously long compared to the 
rest and splits just those at their single largest internal gap.

**Key Concept:** Flag any window longer than 1.5× the median delivery duration as a likely 
merge, then split it at the midpoint of its largest internal gap in ball detection.

**Hyperparameters:** `DURATION_SPLIT_THRESHOLD = median_dur * 1.5` — derived from the data 
itself rather than a fixed frame count, so it adapts to this video's actual delivery lengths.

In [12]:
durations = np.array([e - s for s, e in deliveries])
median_dur = np.median(durations)
DURATION_SPLIT_THRESHOLD = median_dur * 1.5  # flag windows well beyond typical length

print(f"Median delivery duration: {median_dur:.0f} frames")
print(f"Flagging windows longer than: {DURATION_SPLIT_THRESHOLD:.0f} frames\n")

final_deliveries = []

for (s, e) in deliveries:
    duration = e - s
    if duration <= DURATION_SPLIT_THRESHOLD:
        final_deliveries.append((s, e))
        continue

    # Find the largest internal gap in this window and split there
    segment = ball_present[s:e]
    gaps = []  # (gap_length, gap_start_idx_in_segment)
    current_gap = 0
    gap_start = None
    for i, present in enumerate(segment):
        if not present:
            if current_gap == 0:
                gap_start = i
            current_gap += 1
        else:
            if current_gap > 0:
                gaps.append((current_gap, gap_start))
            current_gap = 0
    if current_gap > 0:
        gaps.append((current_gap, gap_start))

    if not gaps:
        final_deliveries.append((s, e))  # no gap found, leave as-is
        continue

    largest_gap_len, largest_gap_start = max(gaps)
    split_point = s + largest_gap_start + largest_gap_len // 2  # split mid-gap

    print(f"Splitting window {s}-{e} ({duration}f) at frame {split_point} "
          f"(gap of {largest_gap_len} frames)")
    final_deliveries.append((s, split_point))
    final_deliveries.append((split_point, e))

print(f"\nFinal delivery count: {len(final_deliveries)}")
for i, (s, e) in enumerate(final_deliveries):
    print(f"  Delivery {i+1}: frames {s}-{e} ({e-s} frames, {(e-s)/VIDEO_FPS:.2f}s)")

Median delivery duration: 102 frames
Flagging windows longer than: 153 frames

Splitting window 320-509 (189f) at frame 484 (gap of 7 frames)
Splitting window 654-836 (182f) at frame 755 (gap of 12 frames)
Splitting window 1255-1458 (203f) at frame 1362 (gap of 9 frames)
Splitting window 3444-3606 (162f) at frame 3582 (gap of 14 frames)

Final delivery count: 23
  Delivery 1: frames 40-144 (104 frames, 3.47s)
  Delivery 2: frames 169-263 (94 frames, 3.13s)
  Delivery 3: frames 320-484 (164 frames, 5.47s)
  Delivery 4: frames 484-509 (25 frames, 0.83s)
  Delivery 5: frames 654-755 (101 frames, 3.37s)
  Delivery 6: frames 755-836 (81 frames, 2.70s)
  Delivery 7: frames 940-1026 (86 frames, 2.87s)
  Delivery 8: frames 1137-1189 (52 frames, 1.73s)
  Delivery 9: frames 1255-1362 (107 frames, 3.57s)
  Delivery 10: frames 1362-1458 (96 frames, 3.20s)
  Delivery 11: frames 1479-1558 (79 frames, 2.63s)
  Delivery 12: frames 1581-1659 (78 frames, 2.60s)
  Delivery 13: frames 1682-1767 (85 frames

### Cell 10c: Filter Out Spurious Tiny Fragments After Splitting

**Why this cell:** Splitting long windows in Cell 10b can produce a real second delivery, but 
sometimes only a short spurious tail (a stray false-positive detection after the real delivery 
ended). Genuine deliveries are all 52+ frames, so any fragment shorter than that is discarded 
rather than counted as its own delivery.

**Key Concept:** Simple length filter applied after splitting, using the shortest real delivery 
observed so far as the cutoff.

**Hyperparameters:** `MIN_DELIVERY_LENGTH = 50` frames — just below the smallest confirmed 
genuine delivery (52 frames), so it only removes clearly spurious fragments.

In [13]:
MIN_DELIVERY_LENGTH = 50  # smallest genuine delivery seen so far is 52 frames

filtered_deliveries = [(s, e) for (s, e) in final_deliveries if (e - s) >= MIN_DELIVERY_LENGTH]

print(f"Before filtering: {len(final_deliveries)} windows")
print(f"After filtering (>= {MIN_DELIVERY_LENGTH} frames): {len(filtered_deliveries)} windows\n")

for i, (s, e) in enumerate(filtered_deliveries):
    print(f"  Delivery {i+1}: frames {s}-{e} ({e-s} frames, {(e-s)/VIDEO_FPS:.2f}s)")

Before filtering: 23 windows
After filtering (>= 50 frames): 21 windows

  Delivery 1: frames 40-144 (104 frames, 3.47s)
  Delivery 2: frames 169-263 (94 frames, 3.13s)
  Delivery 3: frames 320-484 (164 frames, 5.47s)
  Delivery 4: frames 654-755 (101 frames, 3.37s)
  Delivery 5: frames 755-836 (81 frames, 2.70s)
  Delivery 6: frames 940-1026 (86 frames, 2.87s)
  Delivery 7: frames 1137-1189 (52 frames, 1.73s)
  Delivery 8: frames 1255-1362 (107 frames, 3.57s)
  Delivery 9: frames 1362-1458 (96 frames, 3.20s)
  Delivery 10: frames 1479-1558 (79 frames, 2.63s)
  Delivery 11: frames 1581-1659 (78 frames, 2.60s)
  Delivery 12: frames 1682-1767 (85 frames, 2.83s)
  Delivery 13: frames 1788-1923 (135 frames, 4.50s)
  Delivery 14: frames 2040-2132 (92 frames, 3.07s)
  Delivery 15: frames 2181-2277 (96 frames, 3.20s)
  Delivery 16: frames 2483-2585 (102 frames, 3.40s)
  Delivery 17: frames 2686-2810 (124 frames, 4.13s)
  Delivery 18: frames 2971-3080 (109 frames, 3.63s)
  Delivery 19: frames 

### Cell 11: Loop track_and_measure_delivery() Over All Detected Deliveries

**Why this cell:** With `filtered_deliveries` giving us 21 validated frame windows, we can now 
run the same tracking + speed logic from Cell 9 on every real delivery in the video, collecting 
one summary row per delivery instead of manually re-running Cell 7/8/9 by hand each time.

**Key Concept:** Call `track_and_measure_delivery()` once per window, collect each summary dict 
into a list, then build a single results table. Individual `traj_df`s are kept in a dict too, 
in case we want to inspect any one delivery's raw trajectory later (e.g. Delivery 3, flagged 
above as worth double-checking).

**Hyperparameters:** Same defaults as Cell 9 — no new tunables here, just orchestration.

In [14]:
all_summaries = []
all_trajectories = {}

for i, (s, e) in enumerate(filtered_deliveries):
    traj_df, summary = track_and_measure_delivery(
        video_path=VIDEO_PATH,
        detector=detector,
        start_frame=s,
        end_frame=e,
        meters_per_pixel=METERS_PER_PIXEL,
        video_fps=VIDEO_FPS,
        max_ball_speed_kmh=MAX_BALL_SPEED_KMH,
    )
    summary["delivery_id"] = i + 1
    all_summaries.append(summary)
    all_trajectories[i + 1] = traj_df
    print(f"Delivery {i+1:2d} ({s}-{e}): peak={summary['peak_speed_kmh']}, "
          f"mean={summary['mean_speed_kmh']}, detected={summary['frames_detected']}/{summary['frames_tracked']}")

results_df = pd.DataFrame(all_summaries)[
    ["delivery_id", "start_frame", "end_frame", "frames_tracked", "frames_detected",
     "peak_speed_kmh", "mean_speed_kmh", "peak_speed_smooth_kmh"]
]
results_df

Delivery  1 (40-144): peak=73.43563531255107, mean=37.9582157195593, detected=97/104
Delivery  2 (169-263): peak=63.26294876312933, mean=44.91664902392439, detected=88/94
Delivery  3 (320-484): peak=62.97664290452517, mean=23.984616531074202, detected=161/164
Delivery  4 (654-755): peak=60.4132912700618, mean=35.241844966512325, detected=94/101
Delivery  5 (755-836): peak=80.40697392009574, mean=51.439686498713094, detected=75/75
Delivery  6 (940-1026): peak=67.16166040053129, mean=44.2634512221352, detected=86/86
Delivery  7 (1137-1189): peak=66.09062881378901, mean=44.15389993116594, detected=48/52
Delivery  8 (1255-1362): peak=105.29199674155797, mean=31.290422257827306, detected=97/107
Delivery  9 (1362-1458): peak=57.709783805765106, mean=37.1194733586921, detected=91/91
Delivery 10 (1479-1558): peak=64.29558751071012, mean=47.70100863112067, detected=78/79
Delivery 11 (1581-1659): peak=60.71618485330257, mean=34.79151205634085, detected=76/78
Delivery 12 (1682-1767): peak=79.2685

,delivery_id,start_frame,end_frame,frames_tracked,frames_detected,peak_speed_kmh,mean_speed_kmh,peak_speed_smooth_kmh
0,1,40,144,104,97,73.435635,37.958216,83.916972
1,2,169,263,94,88,63.262949,44.916649,85.195106
2,3,320,484,164,161,62.976643,23.984617,62.366971
3,4,654,755,101,94,60.413291,35.241845,60.041982
4,5,755,836,75,75,80.406974,51.439686,79.361086
5,6,940,1026,86,86,67.161660,44.263451,66.383999
6,7,1137,1189,52,48,66.090629,44.153900,65.438089
7,8,1255,1362,107,97,105.291997,31.290422,63.780634
8,9,1362,1458,91,91,57.709784,37.119473,56.079558
9,10,1479,1558,79,78,64.295588,47.701009,63.427758


### Cell 12a: Truncate Delivery Windows When Ball Reaches Batsman's Wicket

**Why this cell:** The detector keeps tracking the ball after bat contact too, since it has no 
concept of "hit" — just "ball visible." This inflates delivery duration and drags mean speed 
down with post-hit motion that isn't part of the actual delivery. Since we only care about 
release/flight speed, each delivery window should end as soon as the ball's tracked position 
reaches the striker's (batsman's) wicket — physically, that's when the delivery is over.

**Key Concept:** Direction of travel is determined from the first few raw detections (robust to 
post-hit noise at the end of the window), then the window is truncated the first frame the ball's 
KF x-position comes within `proximity_px` of the wicket it's heading toward.

**Hyperparameters:** `WICKET_PROXIMITY_PX = 200` — widened from an initial 80px after observing 
that batsmen intercept the ball 85-168px in front of the wicket center, not at it. 
`MIN_FRAMES_BEFORE_CUTOFF = 5` — avoids truncating immediately at release.

In [15]:
w1_x, w2_x = w1_mean[0], w2_mean[0]  # from Cell 6 calibration

WICKET_PROXIMITY_PX = 200
MIN_FRAMES_BEFORE_CUTOFF = 5  # ignore the first few frames so we don't cut immediately near release

def truncate_at_batsman_wicket_v2(traj_df, w1_x, w2_x, proximity_px, min_frames_before_cutoff):
    valid_raw = traj_df.dropna(subset=["raw_x"])
    if len(valid_raw) < 2:
        return traj_df, None, "insufficient_data"

    start_x = valid_raw["raw_x"].iloc[:3].mean()

    dist_to_w1 = abs(start_x - w1_x)
    dist_to_w2 = abs(start_x - w2_x)
    target_x = w2_x if dist_to_w1 < dist_to_w2 else w1_x
    heading_positive = target_x > start_x

    cutoff_idx = None
    for i, row in traj_df.iterrows():
        if i < min_frames_before_cutoff:
            continue
        x = row["kf_x"]
        if heading_positive and x >= target_x - proximity_px:
            cutoff_idx = i
            break
        if not heading_positive and x <= target_x + proximity_px:
            cutoff_idx = i
            break

    if cutoff_idx is None:
        closest_approach = (traj_df["kf_x"].max() if heading_positive else traj_df["kf_x"].min())
        gap_remaining = abs(target_x - closest_approach)
        return traj_df, None, f"never_reached (closest approach {gap_remaining:.0f}px short)"

    return traj_df.loc[:cutoff_idx].reset_index(drop=True), int(traj_df.loc[cutoff_idx, "frame"]), "truncated"

### Cell 12b: Apply Truncation to Every Delivery's Trajectory

**Why this cell:** Cell 12 only defined the truncation function — this actually runs it across 
every delivery's stored trajectory from Cell 11, replacing the full (untruncated) trajectories 
with release-only windows, and recomputing peak/mean speed from the truncated data instead of 
the original.

**Key Concept:** Loop over `all_trajectories`, call `truncate_at_batsman_wicket_v2()` per 
delivery, and rebuild summary statistics (peak/mean speed) from only the truncated portion — so 
post-hit motion no longer drags the numbers down.

**Hyperparameters:** None new — reuses `WICKET_PROXIMITY_PX` and `MIN_FRAMES_BEFORE_CUTOFF` 
from Cell 12.

In [16]:
truncated_trajectories = {}
truncated_summaries = []

for delivery_id, df in all_trajectories.items():
    trunc_df, cutoff_frame, status = truncate_at_batsman_wicket_v2(
        df, w1_x, w2_x, WICKET_PROXIMITY_PX, MIN_FRAMES_BEFORE_CUTOFF
    )
    truncated_trajectories[delivery_id] = trunc_df

    valid_speeds = trunc_df.loc[trunc_df["speed_valid"], "speed_kmh"] if "speed_valid" in trunc_df else pd.Series(dtype=float)

    truncated_summaries.append({
        "delivery_id": delivery_id,
        "original_frames": len(df),
        "truncated_frames": len(trunc_df),
        "cutoff_frame": cutoff_frame,
        "status": status,
        "peak_speed_kmh": float(valid_speeds.max()) if len(valid_speeds) else None,
        "mean_speed_kmh": float(valid_speeds.mean()) if len(valid_speeds) else None,
    })

truncated_results_df = pd.DataFrame(truncated_summaries)
truncated_results_df

,delivery_id,original_frames,truncated_frames,cutoff_frame,status,peak_speed_kmh,mean_speed_kmh
0,1,104,44,83,truncated,57.632440,41.811964
1,2,94,35,203,truncated,63.262949,53.315338
2,3,164,39,358,truncated,62.976643,50.121997
3,4,101,40,693,truncated,60.413291,49.077462
4,5,75,40,800,truncated,59.141758,48.984063
5,6,86,37,976,truncated,67.161660,52.561419
6,7,52,38,1174,truncated,66.090629,50.930091
7,8,107,36,1290,truncated,63.950377,53.621350
8,9,91,43,1409,truncated,56.021817,45.023457
9,10,79,35,1513,truncated,63.587956,55.535924


### Cell 13: Generate Final Annotated Video with Delivery Number, Current/Peak/Avg Speed

**Why this cell:** Produces the final human-reviewable output — every frame annotated with 
detection boxes, which delivery is active, and live/peak/average speed for that delivery, 
positioned in a clear area of the frame (upper-right hillside/sky region) so it never occludes 
the pitch, players, or ball.

**Key Concept:** Single sequential pass through the video. Per-frame speed comes from 
`truncated_trajectories`; peak/average per delivery comes from `truncated_results_df`, computed 
once rather than recalculated live.

**Hyperparameters:** None new — overlay position (`BOX_X1`, etc.) tuned visually against actual 
footage rather than derived from data.

In [17]:
OUTPUT_VIDEO_PATH_V2 = os.path.join(WORKING_DIR, "annotated_deliveries_truncated.mp4")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH_V2, fourcc, VIDEO_FPS, (VIDEO_W, VIDEO_H))

frame_lookup = {}
for delivery_id, df in truncated_trajectories.items():
    for _, row in df.iterrows():
        frame_lookup[int(row["frame"])] = (
            delivery_id,
            row["speed_kmh_smooth"] if pd.notna(row["speed_kmh_smooth"]) else None,
        )

delivery_stats = {
    row["delivery_id"]: (row["peak_speed_kmh"], row["mean_speed_kmh"])
    for _, row in truncated_results_df.iterrows()
}

FONT = cv2.FONT_HERSHEY_SIMPLEX

# Upper-right clear zone (hillside/sky area, based on your marked screenshot)
BOX_X1 = int(VIDEO_W * 0.60)
BOX_Y1 = int(VIDEO_H * 0.15)
BOX_X2 = int(VIDEO_W * 0.98)
BOX_Y2 = int(VIDEO_H * 0.38)

cap = cv2.VideoCapture(VIDEO_PATH)
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    r = detector.predict(frame, imgsz=1280, conf=0.25, verbose=False)[0]
    for b in r.boxes:
        cls_id = int(b.cls.item())
        x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
        color = (0, 255, 0) if cls_id == 2 else (255, 0, 0)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

    # Semi-transparent box in the upper-right clear zone
    overlay = frame.copy()
    cv2.rectangle(overlay, (BOX_X1, BOX_Y1), (BOX_X2, BOX_Y2), (0, 0, 0), -1)
    frame = cv2.addWeighted(overlay, 0.6, frame, 0.4, 0)

    text_x = BOX_X1 + 20
    line1_y = BOX_Y1 + 40
    line2_y = BOX_Y1 + 80
    line3_y = BOX_Y1 + 115

    if frame_idx in frame_lookup:
        delivery_id, current_speed = frame_lookup[frame_idx]
        peak_speed, mean_speed = delivery_stats.get(delivery_id, (None, None))

        current_text = f"Current: {current_speed:.1f} km/h" if current_speed is not None else "Current: --"
        peak_text = f"Peak: {peak_speed:.1f} km/h" if peak_speed is not None else "Peak: --"
        mean_text = f"Avg: {mean_speed:.1f} km/h" if mean_speed is not None else "Avg: --"

        cv2.putText(frame, f"Delivery {delivery_id}", (text_x, line1_y),
                    FONT, 0.9, (0, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(frame, current_text, (text_x, line2_y),
                    FONT, 0.75, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(frame, f"{peak_text} | {mean_text}", (text_x, line3_y),
                    FONT, 0.65, (255, 255, 255), 2, cv2.LINE_AA)
    else:
        cv2.putText(frame, "No active delivery", (text_x, line2_y),
                    FONT, 0.8, (200, 200, 200), 2, cv2.LINE_AA)

    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()

print(f"Annotated video saved to: {OUTPUT_VIDEO_PATH_V2}")

Annotated video saved to: /kaggle/working/annotated_deliveries_truncated.mp4
